# 07 · Políticas, montos y suficiencia de prestaciones, ENASEM 2001–2021

**Pregunta central de la autora, sin modificaciones:** ¿Qué tanto es consistente la expansión de las pensiones y los apoyos públicos con una reducción de las brechas de cobertura y de monto de la protección económica de los adultos mayores en México (ENASEM 2001–2021), qué grupos siguen sin vía propia de protección y cómo cambia esta lectura al considerar a quienes mueren o dejan de ser entrevistados?

La expansión pudo reducir brechas de acceso sin igualar montos o garantizar suficiencia: es una hipótesis por contrastar. La mortalidad y la selección son sensibilidades del eje económico.

**Objetivo independiente:** auditar importes, actualizar poder adquisitivo con INPC oficial, comparar prestaciones parciales con LPI históricas y contextualizar patrones con políticas verificadas. Se reutilizan 06 e integración, sin modificar 01–06, fuentes originales, bases procesadas ni LaTeX. No se rehace el EDA completo.

Principal: personas vivas de 60+ en cada visita, **entrevistas directas**. Sustitutos válidos son sensibilidad. Muestra no ponderada, sin causalidad ni estimación de cobertura nacional. K no es automáticamente contributiva; A no es necesariamente efectivo o una pensión de vejez homogénea.

La fuente sigue el patrón sin outputs del proyecto. [Abrir copia ejecutada con tablas y figuras](../data/interim/notebooks_ejecutados/07_POLITICAS_MONTOS_SUFICIENCIA_ejecutado.ipynb). Los resultados seleccionados están también en el [informe general](../resultados/INFORME_GENERAL.md).

In [ ]:
import json
import pandas as pd
from IPython.display import display, Markdown, Image
from tesis_enasem.monetary_policy import analyze, OUTPUT, verify_inputs, DIMENSIONS
from tesis_enasem.monetary_policy_outputs import amount, conclusions
from tesis_enasem.project_paths import RESULTS_DIR
verify_inputs()
r=analyze()  # solo ampliación07; no rehace01–06
assert r==json.loads((OUTPUT/'resultados.json').read_text()),'Resultados07 no reproducidos'
audit=json.loads((OUTPUT/'auditoria.json').read_text())
figs=json.loads((OUTPUT/'figuras.json').read_text())
policies=json.loads((OUTPUT/'cronologia_politicas.json').read_text())
pd.set_option('display.max_columns',None)
pd.set_option('display.max_rows',100)
pd.set_option('display.float_format',lambda x:f'{x:,.3f}')
def tabla(rows):display(pd.json_normalize(rows,sep='.'))
def figura(key):
    f=next(x for x in figs if x['clave']==key)
    display(Image(filename=str(RESULTS_DIR/f['archivo'])))
    display(Markdown('**Nota:** '+f['nota']))
def resumen(z):
    return dict(ronda=z['ronda'],via=z['via'],grupo=z['grupo'],N_receptores=z['N_receptores'],N_monto=z['N_monto'],
        N_ND=z['N_receptores_sin_monto'],N_real=z['persona']['real']['N'],
        nominal_q25=z['persona']['nominal']['q25'],nominal_mediana=z['persona']['nominal']['mediana'],
        nominal_q75=z['persona']['nominal']['q75'],real_q25=z['persona']['real']['q25'],
        real_mediana=z['persona']['real']['mediana'],real_q75=z['persona']['real']['q75'])


## 1. Auditoría monetaria previa

Nombres exactos, rama beneficiaria, unidad, periodicidad, NR/NS y cobertura de montos antes de comparar precios.

In [ ]:
tabla(audit['campos_originales_disponibles']); tabla(audit['variables_originales']); tabla(audit['receptores']); tabla([{'via':k,**v} for k,v in audit['eventos'].items()])

**Interpretación y límites:** 96 campos monetarios originales disponibles,81 con eventos captados, y 198 alternativas de archivos imputados inventariadas. Los 15 campos sin evento no se confunden con importe 0. Validez y cobertura se presentan por evento/concepto y beneficiario 60+directo, además de cobertura por vía. No equivalen a 96 conceptos homologados. Se conserva cada evento/slot y se enlaza a su beneficiario real, LEFT muchos:uno; no se replica monto por institución. Un evento no es una persona ni su ingreso total. Suma parcial con min_count=1 conserva ND.

## 2. Original, imputado, construido y cero

Versiones alternativas consultables, sin sumar ingresos ni imputar para este análisis.

In [ ]:
tabla([{k:x[k] for k in ['columna','fuente_id','ano','etiqueta','uso07']} for x in audit['alternativas_imputadas']]); display(Markdown(audit['cero_A'])); display(Markdown(audit['imputacion']))

**Interpretación y límites:** ENASEMsimple no contiene un ingreso pensionario total comparable validado. Archivos de imputación son unidades de hogar con campos por rama y versiones; no se adoptan sus escalas o cantidades como ingresos independientes. A 2021 permite cero si no logra valorar especie; se conservan ceros y se compara sensibilidad positiva, sin etiquetarlos automáticamente como cero efectivo.

## 3. INPC oficial y actualización

Referencia septiembre 2026=146.075, base segunda quincena julio 2018=100, serie 865541 actualizada 2024. Consulta 14 octubre 2026. Último mes mensual publicado y verificado.

In [ ]:
tabla([r['fuentes']]); tabla(r['factores'])

**Interpretación y límites:** Se obtuvo la serie oficial INEGI enero 2000–septiembre 2026,321 meses consecutivos, cotejo del último valor con aviso INEGI en DOF 9 octubre 2026. La serie enlazada mantiene base común pese a cambios de canasta/ponderadores. Fórmula nominal×146.075/INPC original. Esto mide poder adquisitivo; no valor presente actuarial de una renta.

## 4. Fecha original y sensibilidad

El cuestionario dice año pasado y mes normal; no registra mes de pago ni calendario de flujos. Ronda nominal no equivale a año real de entrevista.

In [ ]:
tabla(audit['fechas_entrevista']); tabla([dict(ronda=z['ronda'],via=z['via'],N_nominal=z['persona']['nominal']['N'],N_real=z['persona']['real']['N'],**{k:z['persona'][k]['mediana'] for k in ['real','sensibilidad_12m','sensibilidad_ano_entrevista','real_min_fecha','real_max_fecha']}) for z in r['montos'] if z['dimension']=='General'])

**Interpretación y límites:** Principal: precios medios del año calendario anterior al año real de entrevista. Sensibilidad:12 meses anteriores al mes observado, promedio del año de entrevista y extremos mensuales del año previo. Son criterios de valoración, no pagos identificados.2630 directas de ronda 2021 se entrevistaron en 2022; una directa 2012 sin año válido no tiene actualización. No asignar todos los montos a 2012/2018/2021 por nombre de ronda.

## 5. Geografía y LPI

LPI mensual por persona, alimentaria+no alimentaria. Serie histórica CONEVAL 1992–marzo 2025 reestimada 2024; responsabilidad INEGI desde 2025. Los años 2000–2022 están cubiertos.

In [ ]:
tabla(audit['geografia']); tabla(r['comparabilidad'])

**Interpretación y límites:** Corte rural<2500, urbana≥2500, verificado en códigos originales 2012/15 y documentación INEGI 2018/21. urbano corta 100 mil.2001/03 solo≥100 mil puede asignarse urbana; resto indeterminado y ambas líneas como escenarios. Discordancias y falta de fuente quedan ND. LPI sigue periodo del precio principal; actualizar prestación y línea con igual índice no cambia su razón.

## 6. Distribuciones nominales y reales

Todos los N, cuantiles 10/25/50/75/90/95, media, DE e IQR; principales compactados.

In [ ]:
tabla([resumen(z) for z in r['montos'] if z['dimension']=='General']); tabla([{'ronda':z['ronda'],'via':z['via'],'unidad':unit,**z['persona'][unit]} for z in r['montos'] if z['dimension']=='General' for unit in ['nominal','real']]); figura('evolucion')

**Interpretación y límites:** Son montos observados entre receptores, no distribución de recursos en toda la población. No se asigna cero a no receptores, ND o muertos. Diferentes N monetarios por ronda: faltantes entre receptores permanecen visibles. Las líneas punteadas representan solo continuidad condicionada del instrumento, no invariancia garantizada.

## 7. Comparaciones temporales defendibles

Dentro de ronda y ventanas K 2001–03/2012–15/2018–21; A 2003–12/2012–15/2015–18, con límites de slots, captación y composición.

In [ ]:
tabla([{'origen':p['origen'],'destino':p['destino'],'via':via,'N0':amount(r,p['origen'],via)['N_monto'],'N1':amount(r,p['destino'],via)['N_monto'],**{unit+'_cambio_pct':100*(amount(r,p['destino'],via)['persona'][unit]['mediana']/amount(r,p['origen'],via)['persona'][unit]['mediana']-1) for unit in ['nominal','real']}} for p in r['comparabilidad'] for via in ['K','A'] if p[via]])

**Interpretación y límites:** K 2012–15 mediana nominal igual y real menor; A 2012–15 y 2015–18 nominal mayor pero real menor. No se estima un cambio homogéneo de K 2001–21 ni de A 2018–21. Variación de una mediana transversal no es mediana del cambio de las mismas personas. Los extremos temporales y los dos criterios alternativos permiten examinar robustez de dirección.

## 8. Brechas por perfiles

Sexo, edad, escolaridad inicial, localidad 100 mil, ocupación y estado conyugal. Se calculan por separado, sin ajuste multivariado.

In [ ]:
for dim in [x for x in DIMENSIONS if x!='grupo_tipologia']:
    display(Markdown('### '+dim))
    tabla([resumen(z) for z in r['montos'] if z['dimension']==dim])
figura('perfiles2021')
tabla([z for z in r['brechas'] if z['ronda']==2021])

**Interpretación y límites:** Brecha absoluta en pesos y relativa/razón se presentan en ambas unidades. Un factor común reescala diferencias absolutas, sin reducir por sí solo la razón de montos. Factores individuales difieren cuando el año real de entrevista difiere; por eso se calculan antes de resumir. Grupos pequeños, especialmente busca trabajo y A de escolaridad alta en rondas tempranas, requieren cautela.

## 9. Eventos, instituciones y titularidad

Número de eventos, todos capturados válidos frente a sumas parciales, instituciones y tipología de protección; cónyuge como contexto.

In [ ]:
tabla([{'ronda':z['ronda'],'via':z['via'],'N_eventos':z['N_eventos'],'N_eventos_ND':z['N_eventos_ND'],'N_todos_capturados':z['N_todos_eventos_capturados'],'N_parciales':z['N_parciales'],'nominal_evento':z['evento']['nominal']['mediana'],'real_evento':z['evento']['real']['mediana']} for z in r['montos'] if z['dimension']=='General']); tabla(r['evidencia_institucional_K']); tabla([resumen(z) for z in r['montos'] if z['dimension']=='grupo_tipologia']); tabla([x for x in r['conyuge'] if x['dimension']=='General'])

**Interpretación y límites:** Institución multiselección no separa pesos por proveedor; agrupación documental no identifica PAM o Bienestar individualmente. Personas pueden figurar en más de un grupo de evidencia porque reciben varios eventos; no sumar esos N. K no garantiza contributividad. Monto del cónyuge 2018/21 exige pareja confirmada y no es transferencia al focal.33 firmas 2015 no se deduplican automáticamente.

## 10. Cronología verificable y fechas

Cambios normativos federales, antecedentes locales y límites de identificación; cronología en investigación, sin asignar toda A a una política.

In [ ]:
tabla(policies['cronologia']); tabla(policies['por_ronda']); tabla(r['norma_programa']); figura('norma')

**Interpretación y límites:** 70 yMás 2007: 70+, localidades hasta 2500; expansión territorial 2008/09/12. PAM 2013: 65+,525 mensuales;2014–18:580. Bienestar 2019: 1275 y umbral 68,65 indígenas y continuidad de padrón;2020: 1310;2021: 1350 y 1550 tras reforma julio,65+;2022 se documenta únicamente por entrevistas tardías. Inscripción, residencia, presupuesto y fecha de pago no son elegibilidad observada. INAPAM y apoyos agrícolas/becas no son una pensión monetaria homogénea.

## 11. Programa A 5 frente a norma

2021 identifica programa de vejez; otros códigos y A histórico no se asignan automáticamente. Norma mensual ya equivalente al pago bimestral.

In [ ]:
tabla(r['programas_A']); tabla(r['comparacion_A5']); tabla([{'dimension':x['dimension'],'grupo':x['grupo'],'N_K':x['K_evento']['nominal']['N'],'N_A5':x['A5_evento']['nominal']['N'],'K_nominal':x['K_evento']['nominal']['mediana'],'K_real':x['K_evento']['real']['mediana'],'A5_nominal':x['A5_evento']['nominal']['mediana'],'A5_real':x['A5_evento']['real']['mediana'],'diferencia_nominal':x['diferencia_nominal'],'diferencia_real':x['diferencia_real'],'razon_nominal':x['razon_nominal'],'razon_real':x['razon_real']} for x in r['comparacion_K_A5']])

**Interpretación y límites:** 4784 eventos A 5/4784 personas en directas 60+, no necesariamente un beneficio homologado con K. Se compara norma del periodo principal frente a norma del mes de entrevista, con importes nominales/reales y diferencias.20 eventos A 5 en 60–64 indican discordancia respecto de edad normativa; no corregir edades ni borrar registros. No dividir por 2 el monto que ya pidió mensual. Diferencias no prueban incumplimiento u error individual.

## 12. Prestación respecto de umbral

Porcentaje de LPI cubierto, diferencia nominal/real y geografía identificable frente a escenarios.

In [ ]:
tabla([{'ronda':z['ronda'],'via':z['via'],'N_lpi':z['persona']['N_lpi'],'N_geo_ND':z['persona']['N_geografia_ND_monto'],'N_periodo_ND':z['persona']['N_periodo_ND_monto'],'N_umbral_ND':z['persona']['N_umbral_ND_monto'],'ratio_mediana_pct':z['persona']['lpi_pct']['mediana'],'inferior_lpi_pct':z['persona']['prestacion_inferior_lpi_pct'],'diferencia_nominal_mediana':z['persona']['diferencia_lpi_nominal']['mediana'],'diferencia_real_mediana':z['persona']['diferencia_lpi_real']['mediana']} for z in r['montos'] if z['dimension']=='General']); figura('umbral'); tabla([{'dimension':z['dimension'],'grupo':z['grupo'],'via':z['via'],'N':z['persona']['N_lpi'],'mediana_pct':z['persona']['lpi_pct']['mediana'],'q25_pct':z['persona']['lpi_pct']['q25'],'q75_pct':z['persona']['lpi_pct']['q75']} for z in r['montos'] if z['ronda']==2021 and z['dimension']!='grupo_tipologia'])

**Interpretación y límites:** No porcentaje de pobreza. LPI es mensual por persona, pero se compara solo una vía parcial, no ingresos de persona/hogar. No comparar la mediana monetaria total con una línea única y llamarla mediana de razones: primero se calcula cada razón individual.2001/03 geografía asignable selecciona ciudades grandes; no comparar esa muestra con todas localidades de 2012–21 sin escenarios.

## 13. Acceso, monto y umbral en el mismo marco

Se reutiliza 06; la hipótesis no se presupone. Tasas de recepción y montos de receptores tienen denominadores distintos.

In [ ]:
tabla([x for z in r['cobertura06'] if z['ronda']==2021 and z['analisis']=='principal' for x in z['cobertura']]); tabla(r['cobertura_conyugal'])

**Interpretación y límites:** En 2021 sin escolaridad hay mayor tasa de unión propia que con≥10 años, pero mucho menor monto K y mediana A común. Eso no demuestra recursos equivalentes entre todos los integrantes del grupo. Edad 70+ tiene acceso mayor y K menor que 60–64. No ordenar cobertura y monto como si fueran porcentajes de un único universo.

## 14. Selección y sensibilidad actuarial

Cohorte basal 60+, destino sin exigir panel balanceado o edad≥60 nuevamente; muerte, ausencia, sustituto y ND separados.

In [ ]:
tabla([x for x in r['seleccion'] if x['dimension']=='General']); figura('seleccion'); tabla([{'intervalo':f"{x['origen']}–{x['destino']}",**x['grupos'][0]} for x in r['cotas06']])

**Interpretación y límites:** Pares monetarios requieren receptor en ambas visitas y montos válidos: selección aún más estrecha que pares de recepción conocida. Basal frente a retenidos cuantifica diferencias descriptivas, no sesgo corregido. Cambios nominales/reales de las mismas personas pueden diferir de tendencias de medianas transversales. Fallecidos nunca reciben síntomas ni prestaciones imputadas posteriores; ausencia no es muerte.

## 15. Brechas y modalidades en sensibilidad

Evaluar cambios de composición por sexo/edad/escolaridad/localidad/ocupación y sustitutos válidos, sin fusionarlos al principal.

In [ ]:
tabla([x for x in r['seleccion'] if x['dimension'] in ['sexo_grupo','localidad_grupo','educacion_grupo']]); tabla([{'ronda':x['ronda'],'via':x['via'],'N_receptores':x['N_receptores'],'N_sustitutos':x['N_sustitutos'],'nominal':x['persona']['nominal']['mediana'],'real':x['persona']['real']['mediana'],'lpi_pct':x['persona']['lpi_pct']['mediana']} for x in r['sustitutos_sensibilidad']])

**Interpretación y límites:** Sustitutos amplían también el universo, no corrigen los directos. Cotas de cobertura/persistencia y pérdida de observación 06 no son IC. No se implementan Cox, KM, IPW, DiD o causalidad. Mortalidad completa/fecha de evento no disponible para valor presente actuarial.

## 16. Interpretación integrada y estado

Responder si aumentos nominales fueron reales, si brechas monetarias se redujeron y si mayor cobertura implicó mayor recurso suficiente.

In [ ]:
display(Markdown('\n\n'.join(conclusions(r))))

**Interpretación y límites:** Se conserva la pregunta central. La respuesta es parcial y específica por vía/periodo/grupo. Comparabilidad condicionada, faltantes monetarios, valores de especie, ventanas temporales y selección siguen siendo restricciones. Resultados seleccionados se incorporan al informe único y su PDF; no se modifican capítulos ni bases.

In [ ]:
verify_inputs(); assert r==json.loads((OUTPUT/'resultados.json').read_text()); print('Reproducción07 e integridad comprobadas')